---
numbering: false
---

# 5.2. Diagonalization

Recap of eigenvalues and eigenvectors.

$A$ is diagonalizable if and only if it has $n$ linearly independent eigenvectors.

List equivalent conditions.

Writing $A = V \Lambda V^{-1}$ is a way to decompose a matrix into a diagonal matrix of eigenvalues and a matrix of eigenvectors.

Talk about the rank of a matrix and how that relates to its eigenvalues.

Positive semidefinite matrices, and how they relate to the eigenvalues of a matrix. Re-introduce least squares from this perspective.

Somewhere here, talk about these ideas: 

More on symmetric matrices

---

## Symmetric Matrices

Symmetric matrices – that is, square matrices where $A = A^T$ – behave **really** nicely through the lens of eigenvectors. First, two important properties:

1. If $A$ is symmetric, then all of its eigenvalues are real (i.e. none of them are complex numbers).
1. If $A$ is symmetric, then two eigenvectors corresponding to different eigenvalues are orthogonal.

I'm hiding the proofs of these properties **not** because they're optional, but because I'd like you to try them on your own before peeking.

:::{seealso} Proof that symmetric matrices have orthogonal eigenvectors

Suppose $\vec v_1$ and $\vec v_2$ are eigenvectors of the symmetric matrix $A$ corresponding to eigenvalues $\lambda_1$ and $\lambda_2$, respectively, where $\lambda_1 \neq \lambda_2$. This tells us

$$A \vec v_1 = \lambda_1 \vec v_1, \qquad A \vec v_2 = \lambda_2 \vec v_2$$

Then, let's start with $\lambda_1 (\vec v_1 \cdot \vec v_2)$ and see where it gets us.

$$
\begin{align*}
\mu (\vec u \cdot \vec v) &= \mu \vec u \cdot \vec v \\
&= \vec u \cdot (\mu \vec v) \\
&= \vec u \cdot (A \vec v) \\
&= \vec u \cdot (\lambda \vec v) \\
:::

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_unit_circle_and_transform(S, name="Matrix", show_eig=True, return_fig=False):
    # Generate unit circle points
    theta = np.linspace(0, 2 * np.pi, 300)
    circle = np.vstack((np.cos(theta), np.sin(theta))).T
    # Transformed circle (ellipse)
    ellipse = (S @ circle.T).T

    # Compute real eigenvalues and eigenvectors (for drawing axes)
    try:
        eigvals, eigvecs = np.linalg.eig(S)
    except np.linalg.LinAlgError:
        eigvals, eigvecs = None, None
    # Only retain real-valued eigenvectors for the plot
    real_mask = np.abs(np.imag(eigvals)) < 1e-8
    real_eigvecs = np.real(eigvecs[:, real_mask])
    real_eigvals = np.real(eigvals[real_mask])
    # Normalize eigenvectors for plotting (unit)
    if real_eigvecs.shape[1] > 0:
        eigvec_norms = np.linalg.norm(real_eigvecs, axis=0)
        eigvecs_dir = real_eigvecs / eigvec_norms
    else:
        eigvecs_dir = np.zeros((2,0))

    # Set up figure
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(r"$$\text{Unit Circle}$$", fr"$${name}\ \text{{transforms the circle}}$$"),
        horizontal_spacing=0.08
    )

    # Common axis settings: [-2.5,2.5] for each
    axis_style = dict(
        showgrid=True,
        gridcolor="#f0f0f0",
        zeroline=False,
        showline=True,
        linecolor="#f0f0f0",
        mirror=True,
        ticks="outside",
        showticklabels=True,
        tickfont=dict(family="Palatino, serif", size=14),
    )
    for i in [1, 2]:
        fig.update_xaxes(
            range=[-2.5, 2.5],
            constrain="domain",
            dtick=1,
            **axis_style,
            row=1, col=i
        )
        fig.update_yaxes(
            range=[-2.5, 2.5],
            scaleanchor=f"x{i}",
            dtick=1,
            **axis_style,
            row=1, col=i
        )

    # Grid zero lines under all objects
    for i in [1,2]:
        fig.add_shape(
            type="line",
            x0=-2.5, x1=2.5, y0=0, y1=0,
            line=dict(color="rgba(170,170,170,0.25)", width=2, dash="solid"),
            row=1, col=i,
            layer="below"
        )
        fig.add_shape(
            type="line",
            x0=0, x1=0, y0=-2.5, y1=2.5,
            line=dict(color="rgba(170,170,170,0.25)", width=2, dash="solid"),
            row=1, col=i,
            layer="below"
        )

    # Left: unit circle (blue)
    fig.add_trace(
        go.Scatter(
            x=circle[:,0], y=circle[:,1],
            line=dict(color="#3d81f6", width=3),
            fill="toself",
            fillcolor="rgba(61,129,246,0.08)",
            name="Unit Circle",
            showlegend=False
        ),
        row=1, col=1
    )
    # Right: transformed circle (ellipse, orange)
    fig.add_trace(
        go.Scatter(
            x=ellipse[:,0], y=ellipse[:,1],
            line=dict(color="orange", width=3),
            fill="toself",
            fillcolor="rgba(255,140,0,0.10)",
            name="Transformed Circle",
            showlegend=False
        ),
        row=1, col=2
    )

    # Plot dotted lines for the (real) eigenvector directions
    scale = 2.2
    if show_eig and eigvecs_dir.shape[1] > 0:
        for i in range(eigvecs_dir.shape[1]):
            v = eigvecs_dir[:,i]
            # Both directions (±v); lines will extend through the origin
            for sign in [+1, -1]:
                fig.add_trace(
                    go.Scatter(
                        x=[-scale * v[0], scale * v[0]],
                        y=[-scale * v[1], scale * v[1]],
                        mode="lines",
                        line=dict(color="mediumvioletred", width=2, dash="dot"),
                        name="Eigenvector",
                        hoverinfo="skip",
                        showlegend=False
                    ),
                    row=1, col=1
                )
                # Same (transformed) axis on the right: direction is S*v
                v_trans = S @ v
                v_trans_norm = np.linalg.norm(v_trans)
                if v_trans_norm > 1e-10:
                    v_trans_unit = v_trans / v_trans_norm
                    fig.add_trace(
                        go.Scatter(
                            x=[-scale * v_trans_unit[0], scale * v_trans_unit[0]],
                            y=[-scale * v_trans_unit[1], scale * v_trans_unit[1]],
                            mode="lines",
                            line=dict(color="mediumvioletred", width=2, dash="dot"),
                            name="Transformed Eigenvector",
                            hoverinfo="skip",
                            showlegend=False
                        ),
                        row=1, col=2
                    )

    fig.update_layout(
        font=dict(family="Palatino, serif", size=16),
        plot_bgcolor="white",
        paper_bgcolor="white",
        margin=dict(l=20, r=20, t=40, b=20),
        width=800, height=400,
    )

    if return_fig:
        return fig
    fig.show(renderer='png', scale=3)

# Example usage:
A = np.array([[1, 2], [2, 1]])
plot_unit_circle_and_transform(A, name="A")